In [ ]:
dbutils.widgets.text("ticker", "TSLA")
dbutils.widgets.text("start_date", "2023-01-01")
dbutils.widgets.text("end_date", "2023-06-21")

ticker = dbutils.widgets.get("ticker")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"

In [ ]:
import pandas as pd

## Part 1: Venue Ranking Algorithm & Framework

### Overview
Exchange venues vary in their execution quality and liquidity provision. By analyzing
multiple quality metrics across venues for a given security, we can rank them to identify
the venues with the best execution characteristics. This enables traders and market makers
to understand where their orders should be routed for optimal outcomes.

### Ranking Methodology
We rank venues through a multi-metric scoring approach:
1. **Collect daily venue quality metrics** across multiple dimensions (CBBO behavior,
   liquidity provision, fill characteristics)
2. **Normalize each metric independently** by ranking venues from best to worst per metric
3. **Aggregate rankings** by summing individual metric ranks for a composite score
4. **Final ranking** orders venues by composite score (lower rank = better overall venue)

This composite approach ensures venues that excel in one area but are weak in others
will have mid-range scores, reflecting balanced execution quality.

**Note on terminology:**
- **CBBO** (Consolidated Best Bid/Offer) = Best bid/offer across all venues (CBBO is the current standard term, replacing EBBO)
- **BBO** (Best Bid/Offer) = A single venue's best bid/offer (NOT consolidated)

### US Exchange Venues
We analyze execution across major US equity exchanges and alternative trading systems.
These market centers collectively capture nearly all US equity trading volume.

In [ ]:
mics = [
    'XNAS',  # NASDAQ Stock Market
    'XBOS',  # Boston Stock Exchange
    'XPSX',  # Philadelphia Stock Exchange
    'XNYS',  # New York Stock Exchange
    'XASE',  # NYSE American (formerly AMEX)
    'ARCX',  # NYSE Arca
    'BATY',  # Bats BYX
    'BATS',  # Bats BZX
    'EDGA',  # EDGA
    'EDGX',  # EDGX
    'XCIS',  # NYSE Chicago
    'XCHI',  # Chicago Stock Exchange
    'IEXG',  # IEX
    'MEMX',  # MEMX
    'EPRL'   # Pearls
]

### Venue Quality Metrics
BMLL calculates daily venue quality metrics that quantify execution quality across multiple dimensions:

**Consolidated Best Bid/Offer (CBBO) Metrics:**
CBBO is the consolidated best bid/offer across all venues in a region. Note: CBBO is now the standard
term for what was previously called EBBO (European Best Bid Offer).
- *CBBOJoiner*: Orders that join the CBBO across all venues
- *CBBOSetter*: Orders that set the CBBO across all venues
- *TimeAtCBBO*: Duration orders spend at the CBBO
- *ExclusiveTimeAtCBBO*: Time this venue exclusively sets the CBBO

**Liquidity Metrics:**
- *FillProbability*: Likelihood a resting order gets filled (Ask1/Bid1 side)
- *CBBOMeanRestingTime*: Average time orders rest at CBBO before fill
- *TWALiquidityAroundCBBO*: Volume available near CBBO (time-weighted average)

**Trade Flow Metrics:**
- *TradeVolume*: Classification as Lit, Dark, Bilateral, or Non-standard
- *QuoteToTrade*: Relationship between quote updates and trades
- *SweepToFill*: Whether orders sweep across multiple price levels

**Pricing Metrics:**
- *Spread*: Bid-ask spread (both percentage and relative VWA)
- *CBBOOrderPercentage*: Percentage of orders contributing to CBBO

### Ranking Metrics
We select 21 specific metric+suffix combinations that provide comprehensive
execution quality coverage. Each is ranked independently, then summed for composite ranking.

### Metric Mapping to SQL Column Names

When running in Snowflake or Databricks, the ANALYTICS table uses SQL column names
instead of the BMLL API's metric|suffix notation. The mapping below translates
our 21 ranking metrics to their corresponding database columns.
This allows the same ranking algorithm to work across platforms with minimal changes.

In [ ]:
LIQ_GROUP_BY_COLS_SF = {
    # CBBO Joiner metrics: Orders joining the CBBO (bid and ask sides, % and notional)
    # CBBO = Consolidated Best Bid/Offer (same as EBBO - now the standard term)
    'CBBOJoiner|AskTWAPercentage': 'CBBO_JOINER_ASK_TWA_PERCENTAGE',
    'CBBOJoiner|AskTWANotional': 'CBBO_JOINER_ASK_TWA_NOTIONAL',
    'CBBOJoiner|BidTWAPercentage': 'CBBO_JOINER_BID_TWA_PERCENTAGE',
    'CBBOJoiner|BidTWANotional': 'CBBO_JOINER_BID_TWA_NOTIONAL',
    # CBBO Setter metrics: Orders setting the CBBO (bid and ask sides, % and notional)
    'CBBOSetter|AskTWAPercentage': 'CBBO_SETTER_ASK_TWA_PERCENTAGE',
    'CBBOSetter|AskTWANotional': 'CBBO_SETTER_ASK_TWA_NOTIONAL',
    'CBBOSetter|BidTWAPercentage': 'CBBO_SETTER_BID_TWA_PERCENTAGE',
    'CBBOSetter|BidTWANotional': 'CBBO_SETTER_BID_TWA_NOTIONAL',
    # Exclusive time at CBBO: When this venue alone sets the CBBO
    'ExclusiveTimeAtCBBO|Percentage': 'EXCLUSIVE_TIME_AT_CBBO_PERCENTAGE',
    # Fill probability: Likelihood an order at the ask or bid gets filled
    'FillProbability|Ask1': 'FILL_PROBABILITY_ASK1',
    'FillProbability|Bid1': 'FILL_PROBABILITY_BID1',
    # Resting time at CBBO: How long orders rest at the consolidated best
    'CBBOMeanRestingTime|Filled': 'CBBO_MEAN_RESTING_TIME_FILLED',
    'CBBOOrderPercentage|Filled': 'CBBO_ORDER_PERCENTAGE_FILLED',
    # Time at CBBO: Total duration at the consolidated best bid/offer
    'TimeAtCBBO|Percentage': 'TIME_AT_CBBO_PERCENTAGE',
    # Spread relative to volume-weighted average price
    'Spread|RelVWA': 'SPREAD_BPS_VWA',
    # Trade volume by classification: lit vs dark vs bilateral vs non-standard execution
    'TradeVolume|Dark': 'TRADE_VOLUME_DARK',
    'TradeVolume|Lit': 'TRADE_VOLUME_LIT',
    'TradeVolume|Bi': 'TRADE_VOLUME_BI',
    'TradeVolume|Non': 'TRADE_VOLUME_NON',
    # Liquidity depth at 10bps around this venue's best bid/offer (BBO)
    # Note: BBO is this venue's best bid/offer, not the consolidated CBBO
    'TWALiquidityAroundBBO|Ask10bpsNotional': 'LIQUIDITY_AT_DEPTH_BBO_ASK10BPS_NOTIONAL',
    'TWALiquidityAroundBBO|Bid10bpsNotional': 'LIQUIDITY_AT_DEPTH_BBO_BID10BPS_NOTIONAL',
    # Sweep-to-fill: Spread when orders sweep across multiple price levels at 100k USD size
    'SweepToFill|100kUSDTWASpread': 'SWEEP_TO_FILL100K_USD_SPREAD',
}

## Part 2: Ranking Implementation Across Platforms

The `get_ranking()` function implements our multi-metric ranking algorithm across all platforms:

**Step 1: Fetch metrics from BMLL**
- Query all configured metrics across the specified date range and MIC list
- Filter to daily frequency metrics

**Step 2: Aggregate by venue**
- Group data by ListingId (which maps 1:1 to venues/MICs)
- Take mean of each metric across the date range
- This produces a single row per venue with 21 metric values

**Step 3: Rank metrics independently**
- For each metric, rank venues from 1 (best) to N (worst)
- Higher metric values = better, so use descending=False
- NaN values ranked lowest (bottom)

**Step 4: Composite scoring**
- Sum the 21 individual metric ranks per venue
- Lower RankSum = better overall venue

**Step 5: Final ranking**
- Rank venues by RankSum (1 = best, N = worst)
- Add venue identifiers (MIC, Ticker, dates) for context

The result is a ranked table showing each venue's position,
composite score, and date range for reproducibility.

### Databricks Implementation

Databricks implementation uses Apache Spark SQL for efficient distributed computation
of rankings on large datasets. The algorithm is identical to Snowflake; only the
execution engine and table naming conventions differ.

In [ ]:
def get_ranking(ticker, sd, ed):
    """
    Databricks implementation: Rank venues using Spark SQL on ANALYTICS table.

    Fetches pre-computed venue quality metrics from the ANALYTICS table and computes
    rankings using Spark's distributed aggregation and window functions.

    Parameters:
        ticker (str): Stock ticker symbol
        sd (str): Start date in YYYY-MM-DD format
        ed (str): End date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Same schema as notebook version:
                     [ListingId, MIC, Rank, RankSum, Ticker, StartDate, EndDate]

    Note:
        Requires an active Spark session (spark) in the Databricks environment.
        Computes in parallel across Databricks cluster for improved performance on large datasets.
    """
    # Step 1: Build SQL column list from metric mapping
    cols = list(LIQ_GROUP_BY_COLS_SF.values())
    cols_sql = ", ".join(cols)
    mic_list = ", ".join(f"'{m}'" for m in mics)

    # Step 2: Query ANALYTICS table for the date range and venues
    # Spark SQL is executed distributedly on the cluster
    sql = f"""
        SELECT
            LISTING_ID AS ListingId,
            MIC,
            {cols_sql}
        FROM {ANALYTICS_TBL}
        WHERE TICKER = '{ticker}'
          AND MIC IN ({mic_list})
          AND TRADING_DATE BETWEEN '{sd}' AND '{ed}'
    """
    liq = spark.sql(sql).toPandas()

    # Handle empty result - return as-is if no data
    if liq.empty:
        return liq

    # Step 3: Convert volume columns to float for aggregation
    volume_cols = ['TRADE_VOLUME_DARK', 'TRADE_VOLUME_LIT', 'TRADE_VOLUME_BI', 'TRADE_VOLUME_NON']
    for col in volume_cols:
        if col in liq.columns:
            liq[col] = liq[col].astype('float64')

    # Step 4: Aggregate metrics by venue (mean across date range)
    liq_gb = liq.groupby(['ListingId'], as_index=False)[cols].mean()

    # Step 5: Rank each metric independently
    for m in cols:
        liq_gb[f'{m}Rank'] = liq_gb[m].rank(ascending=False, na_option='bottom')

    # Step 6: Sum ranks and compute final ranking
    liq_gb['RankSum'] = liq_gb.iloc[:, -len(cols):].sum(axis=1)
    liq_gb['Rank'] = liq_gb['RankSum'].rank(ascending=True, na_option='bottom')

    # Step 7: Format output with MIC and context
    rank = liq_gb[['ListingId', 'Rank', 'RankSum']].sort_values('RankSum')
    rank = rank.merge(liq[['ListingId', 'MIC']].drop_duplicates(), on='ListingId', how='left')
    rank['Ticker'] = ticker
    rank['StartDate'] = sd
    rank['EndDate'] = ed

    return rank


## Part 3: Single-Ticker Venue Ranking Analysis

Demonstrate venue ranking for a single security to identify the best execution venues.
We analyze Tesla (TSLA) across US venues for the first half of 2023.

**Output columns:**
- **Rank**: Final composite ranking (1 = best venue)
- **RankSum**: Sum of individual metric ranks (lower = better)
- **MIC**: Market Identifier Code (venue/exchange name)
- **ListingId**: BMLL's unique listing identifier
- **Ticker, StartDate, EndDate**: Query parameters for reference

In [ ]:
print(f"\n{'='*80}")
print("SINGLE-TICKER VENUE RANKING: TESLA (TSLA)")
print(f"{'='*80}\n")

rankings = get_ranking('TSLA', '2023-01-01', '2023-06-21')

if len(rankings) > 0:
    print(f"Analyzed {len(rankings)} venues over 6-month period\n")

    # Show top 10 venues
    print(rankings[['MIC', 'Rank', 'RankSum']].head(10).to_string(index=False))

    print(f"\n{'='*80}\n")
    print(f"BEST VENUE: {rankings.iloc[0]['MIC']} (Rank {int(rankings.iloc[0]['Rank'])}, Score {rankings.iloc[0]['RankSum']:.1f})")
    print(f"{'='*80}\n")

## Part 4: Multi-Ticker Comparative Analysis

Extend the analysis across multiple securities to identify venue consistency.
This helps determine which venues are reliably high-quality across the market.

In [ ]:
print(f"\n{'='*80}")
print("MULTI-TICKER COMPARATIVE ANALYSIS")
print(f"{'='*80}\n")

# Analyze multiple major stocks
analysis_tickers = ['AAPL', 'MSFT', 'GOOGL', 'TSLA']
comparison_results = []

for ticker in analysis_tickers:
    print(f"Ranking venues for {ticker}...")
    ticker_rankings = get_ranking(ticker, '2023-01-01', '2023-06-21')

    if len(ticker_rankings) > 0:
        # Get top 5 venues for this ticker
        top_5 = ticker_rankings.head(5)[['MIC', 'Rank', 'RankSum']].copy()
        top_5['Ticker'] = ticker
        comparison_results.append(top_5)

# Combine and display results
if comparison_results:
    print(f"\n{'='*80}")
    print("TOP 5 VENUES BY TICKER")
    print(f"{'='*80}\n")

    for ticker in analysis_tickers:
        ticker_data = next((df for df in comparison_results if (df['Ticker'] == ticker).any()), None)
        if ticker_data is not None:
            print(f"\n{ticker}:")
            print(ticker_data[['MIC', 'Rank', 'RankSum']].to_string(index=False))

    print(f"\n{'='*80}\n")

    # Identify most consistent (frequently appearing) venues
    all_comparison = pd.concat(comparison_results, ignore_index=True)
    venue_appearances = all_comparison['MIC'].value_counts().reset_index()
    venue_appearances.columns = ['MIC', 'Appearances_in_Top5']

    print("VENUE CONSISTENCY - Appearances in Top 5 Across Tickers:")
    print(venue_appearances.to_string(index=False))
    print(f"\n{'='*80}\n")